# Calculate the amplitudes

This notebook is used to calculate the amplitudes of the seismograms.

Agentic AI was used in this notebook.

By Hiroto Bito

In [7]:
# Import necessary libraries
import os
import sys
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

from obspy import read, UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.stream import Stream

parent_dir = '/home/hbito/cascadia_obs_ensemble/utils'
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from data_client import get_waveforms

In [8]:
# Read the data frame
datasets_dir =  '/wd1/hbito_data/data/datasets_all_regions'
path_assigned_picks_df = f'{datasets_dir}/Cascadia_updated_catalog_picks_assignment_ver_3.csv'

# Prepare output CSV path 
output_csv_path = f'{datasets_dir}/Cascadia_updated_catalog_picks_assignment_ver_3_w_amp_test.csv'

# File to save skipped picks
skipped_csv_path = f'{datasets_dir}/calculate_amplitudes_skipped_picks_test.csv'

assigned_picks_df = pd.read_csv(path_assigned_picks_df, index_col=False).copy()
assigned_picks_df.head()

,Unnamed: 0,time_pick,station,phase,timeres,idx,arid,latitude,longitude,depth,Detection Value,time,RMS Residual (s),Num. P,Num. S,picks,slatitude,slongitude,selevation
0,0,2010-01-01T00:15:27.180000Z,UW.PCMD,P,0.049,0,0,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
1,1,2010-01-01T00:15:37.840400Z,UW.RVW,P,1.264,0,1,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.149750,-122.742996,504.0
2,2,2010-01-01T00:15:33.280000Z,UW.PCMD,S,-0.243,0,2,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
3,3,2010-01-01T00:15:42.002000Z,UW.GNW,S,2.402,0,3,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.564130,-122.824980,220.0
4,4,2010-01-01T00:15:43.618400Z,PB.B013,S,-0.651,0,4,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.813000,-122.910797,75.3


In [9]:
# Define the arguments
window_before = 30 # in sec
window_after = 150 # in sec

window_before_calc_amplitude = 0.5 # in sec
window_after_calc_amplitude = 2 # in sec

source = 'pnwstore'

freq_highpass = 2 # in Hz
new_sampling_rate = 100 # in Hz

In [10]:
df = assigned_picks_df.iloc[:100]

In [11]:
# Run the loop
amplitudes = []

for idx, row in tqdm(df.iterrows(), total=len(df)):

    # Define the arguments 
    arid = row['arid']
    orid = row['idx']
    date, _time = row['time'].split(' ')
    datetime_str = date+'T'+_time
    origin_time = UTCDateTime(datetime_str)  # Accept ISO string directly

    time_pick_str = row['time_pick'] 
    time_pick = UTCDateTime(time_pick_str)  # Accept ISO string directly

    network = row['station'].split('.')[0].strip()
    station = row['station'].split('.')[1].strip()
    channel = '*H*'

    starttime = time_pick - window_before 
    endtime = time_pick + window_after

    starttime_trim = time_pick - window_before_calc_amplitude 
    endtime_trim = time_pick + window_after_calc_amplitude

    # Print the number of items in amplitudes for validation
    print('len(amplitudes)',len(amplitudes))    

    # Request a waveform
    time.sleep(0.1)

    try:
        st = get_waveforms(network=network, station=station, channel=channel,
                            starttime=starttime, endtime=endtime,
                            source=source)
    except Exception as e:
        print(f"Request failed: {e}")

        # Save amplitude to the output DataFrame and CSV on the fly
        amp = np.nan
        amplitudes.append(amp)

        # Save skipped info to CSV
        skipped_info = {
            'arid': arid,
            'orid': orid,
            'network': network,
            'station': station,
            'channel': channel,
            'origin_time': origin_time,
            'time_pick': time_pick,
            'starttime': starttime,
            'endtime': endtime,
            'reason': f'Request failed: {e}'
        }
        df_skipped = pd.DataFrame([skipped_info])
        if not os.path.isfile(skipped_csv_path):
            df_skipped.to_csv(skipped_csv_path, mode='w', header=True, index=False)
        else:
            df_skipped.to_csv(skipped_csv_path, mode='a', header=False, index=False)

        continue
        

    # time.sleep(0.1)


    # Create a new stream
    sdata = Stream()
    
    # Check if loaded data have a vertical component (minimum requirement)
    has_Z = bool(st.select(channel='??Z'))
    # Check for the presence of HH, BH, and EH channels
    has_HH = bool(st.select(channel='HH?'))
    has_BH = bool(st.select(channel='BH?'))
    has_EH = bool(st.select(channel='EH?'))

    if not has_Z:
        e = f'No Vertical Component Data Present at {network}.{station} with HHZ, BHZ or EHZ channels at {time_pick_str}. Skipping'
        print(e)

        # Save amplitude in the list
        amp = np.nan
        amplitudes.append(amp)

        # Save skipped info to CSV
        skipped_info = {
            'arid': arid,
            'orid': orid,
            'network': network,
            'station': station,
            'channel': channel,
            'origin_time': origin_time,
            'time_pick': time_pick,
            'starttime': starttime,
            'endtime': endtime,
            'reason': f'Request failed: {e}'
        }
        df_skipped = pd.DataFrame([skipped_info])
        if not os.path.isfile(skipped_csv_path):
            df_skipped.to_csv(skipped_csv_path, mode='w', header=True, index=False)
        else:
            df_skipped.to_csv(skipped_csv_path, mode='a', header=False, index=False)

        continue

    # Apply selection logic based on channel presence
    if has_HH:
        # If all HH, BH, and EH, channels are present, select only HH
        sdata += st.select(channel='HH*')
    elif has_BH:
        # If BH and EH channels are present, select only BH
        sdata += st.select(channel='BH*')
    elif has_EH:
        # If only EH channels are present, select only EH
        # NTS: This may result in getting only vertical component data - EH? is used for PNSN analog stations
        # NTS: This may also be tricky for pulling full day-volumes because the sampling rate shifts for
        #      analog stations due to the remote digitization scheme used with analog stations
        sdata += st.select(channel='EH*')
    else:
        e = f'No data available at {network}.{station} with HHZ, BHZ or EHZ channels at {time_pick_str}. Skipping.'
        print(e)

        # Save amplitude to the output DataFrame and CSV on the fly
        amp = np.nan
        amplitudes.append(amp)

        # Save skipped info to CSV
        skipped_info = {
            'arid': arid,
            'orid': orid,
            'network': network,
            'station': station,
            'channel': channel,
            'origin_time': origin_time,
            'time_pick': time_pick,
            'starttime': starttime,
            'endtime': endtime,
            'reason': f'Request failed: {e}'
        }
        df_skipped = pd.DataFrame([skipped_info])
        if not os.path.isfile(skipped_csv_path):
            df_skipped.to_csv(skipped_csv_path, mode='w', header=True, index=False)
        else:
            df_skipped.to_csv(skipped_csv_path, mode='a', header=False, index=False)
        continue

    # Resample
    sdata.resample(new_sampling_rate)
        
    # Apply highpass filter
    sdata.detrend(type='demean')
    sdata.taper(max_percentage=0.05)
    sdata.filter(type='highpass', freq=freq_highpass)

    trimmed_data = sdata.copy().trim(starttime=starttime_trim, endtime=endtime_trim)

    max_amp = 0
    for tr in trimmed_data:
        max_amp = max(max_amp, abs(tr.data).max())

    amplitudes.append(max_amp)

# Append to DataFrame
df.loc[:,"Amplitude"] = amplitudes

df.to_csv(output_csv_path, index=False)

  0%|          | 0/100 [00:00<?, ?it/s]

len(amplitudes) 0


PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
  1%|          | 1/100 [00:02<03:19,  2.02s/it]PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09


len(amplitudes) 1


  3%|▎         | 3/100 [00:02<01:01,  1.57it/s]

...

 98%|█████████▊| 98/100 [00:26<00:00,  5.27it/s]

len(amplitudes) 97
len(amplitudes) 98


PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
100%|██████████| 100/100 [00:27<00:00,  3.69it/s]

len(amplitudes) 99



/tmp/ipykernel_3654391/3953987420.py:165: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,"Amplitude"] = amplitudes


In [14]:
assigned_picks_df_w_amp = pd.read_csv(output_csv_path, index_col=False)
assigned_picks_df_w_amp

,Unnamed: 0,time_pick,station,phase,timeres,idx,arid,latitude,longitude,depth,Detection Value,time,RMS Residual (s),Num. P,Num. S,picks,slatitude,slongitude,selevation,Amplitude
0,0,2010-01-01T00:15:27.180000Z,UW.PCMD,P,0.049,0,0,47.13396,-122.09098,60.147,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0,528.446789
1,1,2010-01-01T00:15:37.840400Z,UW.RVW,P,1.264,0,1,47.13396,-122.09098,60.147,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.149750,-122.742996,504.0,73.754697
2,2,2010-01-01T00:15:33.280000Z,UW.PCMD,S,-0.243,0,2,47.13396,-122.09098,60.147,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0,897.628533
3,3,2010-01-01T00:15:42.002000Z,UW.GNW,S,2.402,0,3,47.13396,-122.09098,60.147,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.564130,-122.824980,220.0,1671.719173
4,4,2010-01-01T00:15:43.618400Z,PB.B013,S,-0.651,0,4,47.13396,-122.09098,60.147,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.813000,-122.910797,75.3,60.723817
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,2010-01-01T08:52:03.838400Z,PB.B005,P,0.023,3,95,47.97665,-122.90617,20.910,0.756,2010-01-01 08:51:56.196000+00:00,0.465,10,10,20,48.059547,-123.503281,302.7,60.551756
96,96,2010-01-01T08:52:09.048400Z,PB.B011,P,-1.025,3,96,47.97665,-122.90617,20.910,0.756,2010-01-01 08:51:56.196000+00:00,0.465,10,10,20,48.649544,-123.448189,22.0,73.087010
97,97,2010-01-01T08:52:14.310000Z,CN.GOBB,P,0.069,3,97,47.97665,-122.90617,20.910,0.756,2010-01-01 08:51:56.196000+00:00,0.465,10,10,20,48.949300,-123.510500,173.0,55.218456
98,98,2010-01-01T08:52:15.378400Z,PB.B926,P,-0.397,3,98,47.97665,-122.90617,20.910,0.756,2010-01-01 08:51:56.196000+00:00,0.465,10,10,20,48.820202,-124.131203,191.6,18.716581


In [15]:
skipped_picks_df = pd.read_csv(skipped_csv_path, index_col=False)
skipped_picks_df

FileNotFoundError: [Errno 2] No such file or directory: '/wd1/hbito_data/data/datasets_all_regions/calculate_amplitudes_skipped_picks_test.csv'